
# 학습 목표

- `merge`와 `groupby`를 함께 써서 Titanic 열을 다시 읽어봐요.
- `Age` 결측치, `Sex_enc`, `Deck` 전처리를 직접 해봐요.
- `get_dummies(drop_first=True)`와 `pd.concat(axis=1)` 흐름을 손으로 익혀요.
- Telco 데이터에서 인코딩, `train_test_split(stratify=y)`, 스케일링까지 한 번에 연결해봐요.
- 데이터 누수를 막는 순서를 말로 설명할 수 있게 해요.
- 

In [1]:
import pandas as pd
from pathlib import Path

In [2]:

def load_csv(filename):
    candidates = [Path(filename), Path.cwd() / filename]
    candidates.extend(Path.cwd().rglob(filename))

    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return pd.read_csv(path)

    raise FileNotFoundError(f"{filename} 파일을 찾지 못했어요.")

## 문제 1. [쉬움] Titanic 열 두 묶음을 merge해서 다시 보기

### 문제 설명

Titanic에서 `PassengerId`를 key로 써서 가로로 합쳐볼게요.  
기본 정보 표와 요금 표를 따로 만든 뒤 merge하고, 합쳐진 결과에서 `Pclass`별 인원수와 평균 `Fare`를 구해보세요.

In [7]:
# TODO: Titanic 데이터를 읽어와요.
df = pd.read_csv('../../data/Titanic-Dataset.csv')

In [ ]:
# TODO: PassengerId, Pclass, Name이 들어 있는 기본 정보 표를 만들어요.
basic = df[['PassengerId', 'Pclass', 'Name']]
basic.head()

,PassengerId,Pclass,Name
0,1,3,"Braund, Mr. Owen Harris"
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,3,3,"Heikkinen, Miss. Laina"
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,5,3,"Allen, Mr. William Henry"


In [ ]:
# TODO: PassengerId, Fare, Embarked가 들어 있는 요금 표를 만들어요.
price = df[['PassengerId', 'Fare', 'Embarked']]

In [14]:
price.head()

,PassengerId,Fare,Embarked
0,1,7.2500,S
1,2,71.2833,C
2,3,7.9250,S
3,4,53.1000,S
4,5,8.0500,S


In [ ]:
# TODO: PassengerId를 기준으로 inner merge를 해요.

df3=pd.merge(basic, price,on='PassengerId',how='inner')

In [ ]:
# TODO: 합쳐진 표의 앞 5행을 출력해요.
df3.head(5)

,PassengerId,Pclass,Name,Fare,Embarked
0,1,3,"Braund, Mr. Owen Harris",7.2500,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",71.2833,C
2,3,3,"Heikkinen, Miss. Laina",7.9250,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",53.1000,S
4,5,3,"Allen, Mr. William Henry",8.0500,S


In [20]:
# TODO: Pclass별 인원수와 평균 Fare를 groupby로 구하고 reset_index()까지 해요.
df_copy = df.copy()
result = (     
      df_copy
      .groupby("Pclass")
      .agg(passenger_count=("PassengerId", "count"), mean_fare=("Fare", "mean"))
      .reset_index()
  )
print(result)

   Pclass  passenger_count  mean_fare
0       1              216  84.154687
1       2              184  20.662183
2       3              491  13.675550


## 문제 2. [쉬움] Titanic Age, Sex, Cabin 전처리하기

### 문제 설명

이제 Day 2의 핵심 전처리를 직접 해볼게요.  
`Age`는 중앙값으로 채우고, `Sex`는 `Sex_enc`로 바꾸고, `Cabin` 첫 글자를 뽑아서 `Deck`을 만들어 보세요.



In [25]:
# TODO: Titanic 데이터를 읽고 복사본을 만들어요.
titanic = df.copy()

# TODO: 처리 전 Age 결측 개수를 출력해요.
titanic['Age'].isnull().sum()

np.int64(0)

In [26]:
# TODO: Age 중앙값을 변수에 저장해요.
age_median = titanic['Age'].median()

# TODO: Age 결측치를 중앙값으로 채워요.
titanic['Age'] = titanic['Age'].fillna(age_median)

In [28]:
# TODO: Sex를 {"male": 0, "female": 1}로 매핑해서 Sex_enc 열을 만들어요.
titanic['Sex_enc'] = titanic['Sex'].map({'male' : 0, 'female':1})

In [29]:

# TODO: Cabin 첫 글자를 뽑고, 결측은 "U"로 채워 Deck 열을 만들어요.

titanic['Deck'] =titanic['Cabin'].str[0].fillna('U')

In [30]:
## TODO: Age 결측 개수, Sex_enc 결측 개수, Deck value_counts()를 출력해요.

print(titanic['Age'].isnull().sum())
print(titanic['Sex_enc'].isnull().sum())
print(titanic['Deck'].value_counts())

0
0
Deck
U    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64


## 문제 3. [보통] Titanic Embarked, Deck 원핫 인코딩 후 X 만들기

### 문제 설명

앞 문제에서 만든 Titanic 전처리 결과를 이어서 써볼게요.  
`Embarked` 결측은 `S`로 채우고, `Embarked`와 `Deck`은 `get_dummies(drop_first=True)`로 바꾼 뒤, `pd.concat(axis=1)`으로 붙여 최종 X를 만들어 보세요.
### 힌트

- 기본 열은 `["Pclass", "Sex_enc", "Age", "SibSp", "Parch", "Fare"]`를 그대로 써보세요.
- `pd.concat([base_df, embarked_dummies, deck_dummies], axis=1)` 형태예요.
- 기대하는 더미열 이름은 `Emb_C`, `Emb_Q`, `Deck_C`, `Deck_E`예요.

### 완료 확인 기준

- X에 기본 6개 열과 더미 4개 열이 함께 있어야 해요.
- `X.isnull().sum().sum()` 결과가 0이면 성공이에요.
- `pd.concat`에서 행이 아니라 열이 붙었는지 꼭 확인해요.


In [34]:
# TODO: Embarked 결측을 최빈값 "S"로 채워요.
titanic['Embarked' ]=titanic['Embarked'].fillna('S')

In [35]:
titanic.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Sex_enc          0
Deck             0
dtype: int64

In [ ]:

# TODO: Embarked는 categories=["S", "C", "Q"]로 Categorical을 만들어요.
titanic['Embarked'] = pd.Categorical(df['Embarked'], categories=['S','C','Q'])

In [36]:

# TODO: Deck은 categories=["U", "C", "E"]로 Categorical을 만들어요.
titanic['Deck'] = pd.Categorical(titanic['Deck'], categories=['U','C','E'])

C:\Users\main\AppData\Local\Temp\ipykernel_3916\3875813968.py:2: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  titanic['Deck'] = pd.Categorical(titanic['Deck'], categories=['U','C','E'])


In [39]:
# TODO: Embarked 더미열과 Deck 더미열을 각각 get_dummies(drop_first=True, dtype=int)로 만들어요.

embarked_dummies = pd.get_dummies(titanic['Embarked'], prefix='Emb', drop_first=True, dtype=int)
deck_dummies = pd.get_dummies(titanic['Deck'], prefix='Deck', drop_first=True,dtype=int)

In [ ]:
numeric_df = titanic[["Pclass", "Sex_enc", "Age", "SibSp", "Parch", "Fare"]]

In [42]:
# TODO: 숫자 기본 열 6개와 더미열을 axis=1로 concat해요.

base_df = pd.concat([numeric_df, embarked_dummies, deck_dummies], axis=1)


In [44]:
# TODO: X의 열 목록, shape, 결측 수 합계를 출력해요.
print(base_df.columns.tolist())
print(base_df.shape)
print(base_df.isnull().sum().sum())

['Pclass', 'Sex_enc', 'Age', 'SibSp', 'Parch', 'Fare', 'Emb_Q', 'Emb_S', 'Deck_C', 'Deck_E']
(891, 10)
0


## 문제 4. [보통] Telco 범주형 인코딩하기

### 문제 설명

이번에는 Telco로 넘어가 볼게요.  
먼저 `Contract`와 `InternetService`별 Churn 비율을 보고, 그다음 이진 범주형은 `map()`, 다값 범주형은 `get_dummies()`로 인코딩해 보세요.

### 힌트

- `binary_cols`에는 `gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn`이 들어가요.
- `multi_cols`에는 `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `Contract`, `PaymentMethod`가 들어가요.
- `groupby(...).value_counts(normalize=True).rename("ratio").reset_index()` 패턴을 다시 떠올려 보세요.

### 완료 확인 기준

- `Contract` 비율 표와 `InternetService` 비율 표가 각각 보여야 해요.
- `Contract_One year`, `Contract_Two year` 열이 보이면 성공이에요.
- 왜 `Contract`에 레이블 인코딩보다 원핫 인코딩이 더 안전한지도 한 줄로 적어 보세요.


In [78]:
# TODO: Telco 데이터를 읽고 복사본을 만들어요.
telco = pd.read_csv('../../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = telco.copy()

In [79]:
# TODO: Contract별 Churn 비율 표를 만들어요.
contract_churn = (
    df.groupby('Contract')['Churn']
        .value_counts(normalize=True)
        .rename('ratio')
        .reset_index(
        )
)
contract_churn.head()

,Contract,Churn,ratio
0,Month-to-month,No,0.572903
1,Month-to-month,Yes,0.427097
2,One year,No,0.887305
3,One year,Yes,0.112695
4,Two year,No,0.971681


## 비율표를 만드는 목적이 뭘까
- 인코딩 전에 데이터를 먼저 이해하려는 목적이다.
- 탐색적 분석(EDA) 단계이다.
- value_counts(normalize=True) — 각 값의 빈도수를 비율(0~1)로 반환. normalize=False면 개수, True면 비율.
- .rename("ratio") — 비율 열의 이름을 "ratio"로 지정. 없으면 열 이름이 "proportion" 등 기본값으로 나옴.
- .reset_index() — groupby/value_counts로 인덱스가 된 Contract, Churn을 일반 열로 내려줌. 표 형태로 보기 편해짐.

In [80]:
internetService_churn = (
    df.groupby('InternetService')['Churn']
        .value_counts(normalize=True)
        .rename('ratio')
        .reset_index()
    
)
internetService_churn.head()

,InternetService,Churn,ratio
0,DSL,No,0.810409
1,DSL,Yes,0.189591
2,Fiber optic,No,0.581072
3,Fiber optic,Yes,0.418928
4,No,No,0.925950


In [81]:
binary_cols =['gender', 'Partner','Dependents', 'PhoneService', 'PaperlessBilling','Churn']

In [82]:
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity','OnlineBackup', 'DeviceProtection','TechSupport', 'StreamingTV','StreamingMovies','Contract','PaymentMethod']

In [83]:
print(df['gender'].info())
df['gender'].head()

<class 'pandas.Series'>
RangeIndex: 7043 entries, 0 to 7042
Series name: gender
Non-Null Count  Dtype
--------------  -----
7043 non-null   str  
dtypes: str(1)
memory usage: 89.5 KB
None


0    Female
1      Male
2      Male
3      Male
4    Female
Name: gender, dtype: str

In [84]:
# TODO: binary_map 사전을 만들고, binary_cols를 반복하면서 map()을 적용해요.
binary_map = {'Yes':1, 'No':0, 'Male' :1,"Female":0}
for col in binary_cols:
    print(col, df[col].unique())
    df[col] = df[col].map(binary_map)
    print(df[col])

gender <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
0       0
1       1
2       1
3       1
4       0
       ..
7038    1
7039    0
7040    0
7041    1
7042    1
Name: gender, Length: 7043, dtype: int64
Partner <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
0       1
1       0
2       0
3       0
4       0
       ..
7038    1
7039    1
7040    1
7041    1
7042    0
Name: Partner, Length: 7043, dtype: int64
Dependents <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
0       0
1       0
2       0
3       0
4       0
       ..
7038    1
7039    1
7040    1
7041    0
7042    0
Name: Dependents, Length: 7043, dtype: int64
PhoneService <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
0       0
1       1
2       1
3       0
4       1
       ..
7038    1
7039    1
7040    0
7041    1
7042    1
Name: PhoneService, Length: 7043, dtype: int64
PaperlessBilling <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
0       1
1       0
2       1
3       0
4       1

In [85]:
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)

In [86]:
print(len(df.columns))

32


In [87]:
[col for col in df.columns if col.startswith('Contract_')]

['Contract_One year', 'Contract_Two year']

## 문제 5. [도전] Telco를 ML-ready 데이터셋으로 완성하기

### 문제 설명

이제 Day 2를 한 번에 정리해볼게요.  
Telco 데이터를 인코딩한 뒤 y와 X를 나누고, `train_test_split(stratify=y)`를 적용하고, `tenure`, `MonthlyCharges`, `TotalCharges`만 `StandardScaler`로 스케일링해 보세요.
### 힌트

- split을 먼저 하고 스케일링을 나중에 해야 데이터 누수를 막을 수 있어요.
- `X_train.loc[:, num_cols] = ...` 형태로 넣으면 읽기 편해요.
- `y_train.mean()`과 `y_test.mean()`이 비슷하면 `stratify=y`가 잘 들어간 거예요.

### 완료 확인 기준

- `X_train`, `X_test`가 모두 숫자형 중심 표로 준비돼야 해요.
- `X_train`과 `X_test`의 결측 수 합계가 모두 0이어야 해요.
- `y_train.mean()`과 `y_test.mean()`이 둘 다 0.265 근처면 성공이에요.

---

## 제출 전 체크

- [ ] 문제 1: `merge`와 `groupby` 결과를 모두 출력했나요?
- [ ] 문제 2: `Age`, `Sex_enc`, `Deck` 전처리를 끝냈나요?
- [ ] 문제 3: `pd.concat(axis=1)`로 최종 X를 만들었나요?
- [ ] 문제 4: Telco 인코딩 뒤 `Contract_` 더미열을 확인했나요?
- [ ] 문제 5: `stratify=y`와 `fit/transform` 분리를 지켰나요?

## 완료 확인 한 줄 요약

- Titanic은 전처리와 인코딩까지 끝나야 해요.
- Telco는 인코딩과 스케일링까지 끝나야 해요.
- 마지막에는 **"왜 이 순서가 데이터 누수를 막는지"** 를 말로 설명할 수 있으면 진짜 완료예요.


In [98]:
pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   -------------------------------------- - 7.9/8.1 MB 27.1 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 25.1 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------- ----------------------- 14.7/36.6 MB 77.1 MB/s eta 0:00:01
   --------------------------------- ------ 30.7/36.6 MB 78.0 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 66.4 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [sci

In [99]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [101]:
# TODO: y = df["Churn"], X = df.drop(columns=["customerID", "Churn"])로 분리해요.
y=df['Churn']
X = df.drop(columns=['customerID', 'Churn'])

In [111]:
print(X.info())
print('-'*70)
print(X.shape)
X.head()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 30 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   PaperlessBilling                       7043 non-null   int64  
 7   MonthlyCharges                         7043 non-null   float64
 8   TotalCharges                           7043 non-null   str    
 9   MultipleLines_No phone service         7043 non-null   int64  
 10  MultipleLines_Yes                      7043 non-null   int64  
 11  InternetService

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,1,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.5,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,0,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,1,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,0,...,0,0,0,0,0,0,0,0,1,0


In [113]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [114]:
X_train_copy = X_train.copy()

In [115]:
X_test_copy = X_test.copy()

num_cols를 왜 리스트로 나누었는가
- 해당 3개의 열은 나머지 0/1로 인코딩 된거와 달리 매우 큰 값들을 요구한다.
- 그러므로 StandardScaler를 사용해서 머신러닝이 편향이 될 수 있다. 그러므로 StandardScalr를 사용해 평균 0 표준 편차 1로 맞추는것이다.

X_train.loc[:,num_cols] -> 전체행에서 num_cols에 해당하는 열만 추출

In [116]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

In [117]:
scaler = StandardScaler()

In [121]:
X_train["TotalCharges"] = pd.to_numeric(X_train["TotalCharges"], errors="coerce").fillna(0)
X_test["TotalCharges"] = pd.to_numeric(X_test["TotalCharges"], errors="coerce").fillna(0)

In [124]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train[num_cols] = X_train[num_cols].astype(float)
X_test[num_cols] = X_test[num_cols].astype(float)

X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
X_test.loc[:, num_cols] = scaler.transform(X_test[num_cols])

In [125]:
print(X_train.shape, X_test.shape)
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())
print(y_train.mean(), y_test.mean())

(5634, 30) (1409, 30)
0 0
0.2653532126375577 0.2654364797728886
